# Convert prediction JSON to a LaTeX longtable

This notebook selects one document from a result JSON file, aligns its ordered predictions with the sentences containing their verb phrases, and writes a LaTeX `longtable`.

In [24]:
import json
import re
from pathlib import Path
from utils.helpers import find_project_root

PROJECT_ROOT = find_project_root()
DOC_ID = 2
SELECTED_MODEL = "gpt-5.4"
SELECTED_DATASET = "cooking"

To align prediction with source sentence, For each prediction:

1. Begin searching from the last matched sentence.
2. Find the first sentence containing the complete verb phrase, case-insensitively.
3. Assign the prediction to that sentence.
4. Keep the search position at that sentence, allowing multiple predictions to match it.
5. Raise an error if no following sentence contains the verb.


In [25]:
LATEX_ESCAPES = {
    "\\": r"\textbackslash{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "_": r"\_",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
    "^": r"\textasciicircum{}",
}


def escape_latex(value):
    """Escape LaTeX special characters in table content."""
    return "".join(LATEX_ESCAPES.get(char, char) for char in str(value))


def phrase_occurs(phrase, sentence):
    """Return whether a verb phrase occurs as complete words in a sentence."""
    words = str(phrase).split()
    if not words:
        return False
    pattern = r"(?<!\w)" + r"\s+".join(re.escape(word) for word in words) + r"(?!\w)"
    return re.search(pattern, sentence, flags=re.IGNORECASE) is not None


def align_predictions_to_sentences(sentences, predictions):
    """Align ordered predictions to the first matching sentence at or after the current one."""
    aligned = [[] for _ in sentences]
    sentence_index = 0

    for prediction in predictions:
        verb = prediction.get("verb", "")
        match_index = next(
            (
                index
                for index in range(sentence_index, len(sentences))
                if phrase_occurs(verb, sentences[index])
            ),
            None,
        )
        if match_index is None:
            raise ValueError(
                f"Could not align predicted verb {verb!r} at or after sentence "
                f"{sentence_index}."
            )

        aligned[match_index].append(prediction)
        sentence_index = match_index

    return aligned


In [26]:
def json_result_to_longtable(
    input_path,
    doc_id,
    caption,
    label,
):
    """Convert one document in a prediction JSON file to a LaTeX longtable."""
    input_path = Path(input_path)

    with input_path.open(encoding="utf-8") as file:
        documents = json.load(file)

    if isinstance(documents, dict):
        documents = [documents]

    document = next(
        (item for item in documents if str(item.get("doc_id")) == str(doc_id)),
        None,
    )
    if document is None:
        raise ValueError(f"Document {doc_id!r} was not found in {input_path}.")

    sentences = document["sentences"]
    aligned_predictions = align_predictions_to_sentences(
        sentences,
        document.get("prediction", []),
    )

    lines = [
        r"\begin{longtable}{",
        r"    >{\raggedright\arraybackslash}p{0.45\textwidth}",
        r"    >{\raggedright\arraybackslash}p{0.15\textwidth}",
        r"    >{\raggedright\arraybackslash}p{0.30\textwidth}",
        r"}",
        rf"\caption{{{escape_latex(caption)}}}",
        rf"\label{{{label}}} \\",
        "",
        r"% --- HEADER FOR FIRST PAGE ---",
        r"\toprule",
        r"\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\",
        r"\midrule",
        r"\endfirsthead",
        "",
        r"% --- HEADER FOR SUBSEQUENT PAGES ---",
        r"\multicolumn{3}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\",
        r"\toprule",
        r"\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\",
        r"\midrule",
        r"\endhead",
        "",
        r"% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---",
        r"\multicolumn{3}{r}{\textit{Continued on next page}} \\",
        r"\endfoot",
        "",
        r"% --- FOOTER FOR THE LAST PAGE ---",
        r"\bottomrule",
        r"\endlastfoot",
        "",
        r"% --- DATA ---",
        "",
    ]

    for index, (sentence, predictions) in enumerate(
        zip(sentences, aligned_predictions)
    ):
        sentence_text = escape_latex(sentence)
        if not predictions:
            lines.append(
                sentence_text + r" & \textit{--} & \textit{--} \\"
            )
        else:
            for prediction_index, prediction in enumerate(predictions):
                first_column = sentence_text if prediction_index == 0 else ""
                verb_text = escape_latex(prediction.get("verb", ""))
                verb = rf"\(\langle\){verb_text}\(\rangle\)"

                arguments = prediction.get("arguments", [])
                argument_text = "[" + ", ".join(
                    escape_latex(argument) for argument in arguments
                ) + "]"
                lines.append(
                    f"{first_column} & {verb} & {argument_text} " + r"\\"
                )

        if index < len(sentences) - 1:
            lines.extend([r"\midrule", ""])

    lines.extend(["", r"\end{longtable}"])
    latex = "\n".join(lines) + "\n"

    return latex


# NL2P Result

In [27]:
table1 = json_result_to_longtable(
    input_path=PROJECT_ROOT / f"results/nl2p_1/{SELECTED_MODEL}/{SELECTED_DATASET}_nl2p_1_{SELECTED_MODEL}.json",
    doc_id=DOC_ID,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} prompted with NL2P",
    label=f"tab:{SELECTED_DATASET}_preds_nl2p",
)

print(table1)

\begin{longtable}{
    >{\raggedright\arraybackslash}p{0.45\textwidth}
    >{\raggedright\arraybackslash}p{0.15\textwidth}
    >{\raggedright\arraybackslash}p{0.30\textwidth}
}
\caption{Extracted verb-argument predictions by gpt-5.4 prompted with NL2P}
\label{tab:cooking_preds_nl2p} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{3}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---
\multicolumn{3}{r}{\textit{Continued on next page}} \\
\endfoot

% --- FOOTER FOR THE LAST PAGE ---
\bottomrule
\endlastfoot

% --- DATA ---

How to make chocolate raspberry cream pie & \textit{--} & \textit{--} \\
\midrule

How to make this quick dessert & \textit{--} & \textit{--} \\
\midrule

Once you have ga

# NL2P-Refined Result

In [28]:
table2 = json_result_to_longtable(
    input_path=PROJECT_ROOT / f"results/nl2p_1_ablation/{SELECTED_MODEL}/{SELECTED_DATASET}_nl2p_1_ablation_{SELECTED_MODEL}.json",
    doc_id=DOC_ID,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} prompted with NL2P-Refined",
    label=f"tab:{SELECTED_DATASET}_preds_nl2p_refined",
)

print(table2)

\begin{longtable}{
    >{\raggedright\arraybackslash}p{0.45\textwidth}
    >{\raggedright\arraybackslash}p{0.15\textwidth}
    >{\raggedright\arraybackslash}p{0.30\textwidth}
}
\caption{Extracted verb-argument predictions by gpt-5.4 prompted with NL2P-Refined}
\label{tab:cooking_preds_nl2p_refined} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{3}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---
\multicolumn{3}{r}{\textit{Continued on next page}} \\
\endfoot

% --- FOOTER FOR THE LAST PAGE ---
\bottomrule
\endlastfoot

% --- DATA ---

How to make chocolate raspberry cream pie & \textit{--} & \textit{--} \\
\midrule

How to make this quick dessert & \textit{--} & \textit{--} \\
\midrule



# NL2P-Coref Result

In [29]:
table3 = json_result_to_longtable(
    input_path=PROJECT_ROOT / f"results/nl2p_1_coref/{SELECTED_MODEL}/{SELECTED_DATASET}_nl2p_1_coref_{SELECTED_MODEL}.json",
    doc_id=DOC_ID,
    caption=f"Extracted verb-argument predictions by {SELECTED_MODEL} with Corefed Text",
    label=f"tab:{SELECTED_DATASET}_preds_nl2p_coref",
)

print(table3)

\begin{longtable}{
    >{\raggedright\arraybackslash}p{0.45\textwidth}
    >{\raggedright\arraybackslash}p{0.15\textwidth}
    >{\raggedright\arraybackslash}p{0.30\textwidth}
}
\caption{Extracted verb-argument predictions by gpt-5.4 with Corefed Text}
\label{tab:cooking_preds_nl2p_coref} \\

% --- HEADER FOR FIRST PAGE ---
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endfirsthead

% --- HEADER FOR SUBSEQUENT PAGES ---
\multicolumn{3}{c}{\tablename\ \thetable\ -- \textit{Continued from previous page}} \\
\toprule
\textbf{Original Sentence} & \textbf{Verb} & \textbf{Arguments} \\
\midrule
\endhead

% --- FOOTER FOR ALL PAGES EXCEPT THE LAST ---
\multicolumn{3}{r}{\textit{Continued on next page}} \\
\endfoot

% --- FOOTER FOR THE LAST PAGE ---
\bottomrule
\endlastfoot

% --- DATA ---

How to make chocolate raspberry cream pie & \textit{--} & \textit{--} \\
\midrule

How to make this quick dessert & \textit{--} & \textit{--} \\
\midrule

Once you ha